In [ ]:
from pixell import enmap, enplot
from pspy import so_map_preprocessing, pspy_utils
from mnms import utils

import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# for measuring both 2d and 1d spectra (from kspace, no SHTs here)

# the 2d spectra are just measured from the no-hole baseline window. the point
# is to assess the pickup in the actual spectra region, but minimizing the 
# mode-coupling that would come from the holes

# the 1d spectra are meant to be more "cosmological". first we kspace filter
# in the broader kspace mask. then we measure spectra in the baseline mask with
# holes. if we just try to measure spectra and filter in one step/one mask, the
# mode coupling of the pickup swamps the CMB signal. 

so_map_dir = '/scratch/gpfs/SIMONSOBS/users/zatkins/projects/lat-iso/piso/tiger_dr6xbossn_20250804_split/maps/so/chervias'
act_map_dir = '/scratch/gpfs/SIMONSOBS/users/zatkins/projects/lat-iso/piso/tiger_dr6xbossn_20250804_split/maps/act/merrydup/act_projected'
mask_dir = '/scratch/gpfs/SIMONSOBS/users/zatkins/projects/lat-iso/piso/tiger_dr6xbossn_20250804_split/windows'

bin_edges = np.arange(0, 3050, 50) # bins for the 1d cosmo spectra
_bin_edges = np.arange(0, 20000, 50) # bins for the the radial profile flattening of the 2d spectra
lb = (bin_edges[1:] + bin_edges[:-1]) / 2

so_mapset = 'i1_f090'
act_mapset = 'pa5_f090'

so_maps = [enmap.read_map(f'{so_map_dir}/sky_map_{so_mapset}_bundle{i}_20250801.fits') * 1e6 for i in range(4)]
so_maps = enmap.samewcs(so_maps, so_maps[0])
so_kmask = enmap.read_map(f'{mask_dir}/window_so_{so_mapset}_kspace.fits')
so_wmask = enmap.read_map(f'{mask_dir}/window_so_{so_mapset}_baseline.fits')
so_wmask_nh = enmap.read_map(f'{mask_dir}/window_so_{so_mapset}_baseline_no_holes.fits')
so_w2 = np.sum(so_wmask**2 * so_wmask.pixsizemap()) / (4*np.pi)

act_maps = [enmap.read_map(f'{act_map_dir}/act_dr6.02_std_AA_night_{act_mapset}_4way_set{i}_map_srcfree.fits') for i in range(4)]
act_maps = enmap.samewcs(act_maps, act_maps[0])
act_kmask = enmap.read_map(f'{mask_dir}/window_dr6_{act_mapset}_kspace.fits')
act_wmask = enmap.read_map(f'{mask_dir}/window_dr6_{act_mapset}_baseline.fits')
act_wmask_nh = enmap.read_map(f'{mask_dir}/window_dr6_{act_mapset}_baseline_no_holes.fits')
act_w2 = np.sum(act_wmask**2 * act_wmask.pixsizemap()) / (4*np.pi)

modlmap = so_maps.modlmap().astype(np.float32)
lwcs = enmap.lwcs(so_maps.shape, so_maps.wcs)

vk_mask = [-90, 90]
hk_mask = [-50, 50]

fk = so_map_preprocessing.build_std_filter(so_maps.shape, so_maps.wcs, vk_mask=vk_mask, hk_mask=hk_mask, dtype=np.float32)
tf = utils.radial_bin(fk, modlmap, bin_edges)

so_kmaps = enmap.fft(so_maps * so_wmask_nh) # these maps for measuring the pickup in the 2d spectra
act_kmaps = enmap.fft(act_maps * act_wmask_nh)

so_lmaps = enmap.map2harm(enmap.ifft(enmap.fft(so_maps * so_kmask) * fk).real * so_wmask, normalize='phys') # these maps for measuring the 1d cosmological spectra
act_lmaps = enmap.map2harm(enmap.ifft(enmap.fft(act_maps * act_kmask) * fk).real * act_wmask, normalize='phys')

kpols = 'IQU'
lpols = 'TEB'
so_kspecs_2d = {}
act_kspecs_2d = {}
so_lspecs_1d = {}
act_lspecs_1d = {}

# generate all unique pol crosses
for x in range(3):
    for y in range(x, 3):
        kspec = kpols[x] + kpols[y]
        lspec = lpols[x] + lpols[y]

        so_kspecs_2d[kspec] = 0
        act_kspecs_2d[kspec] = 0
        so_lspecs_1d[lspec] = 0
        act_lspecs_1d[lspec] = 0

        # coadd over split-crosses
        for i in range(4):
            for j in range(4):
                if i == j:
                    continue
                
                so_kspecs_2d[kspec] += np.real(so_kmaps[i][x] * so_kmaps[j][y].conj())
                act_kspecs_2d[kspec] += np.real(act_kmaps[i][x] * act_kmaps[j][y].conj())
        
                so_lspecs_1d[lspec] += utils.radial_bin(np.real(so_lmaps[i][x] * so_lmaps[j][y].conj()), modlmap, bin_edges)
                act_lspecs_1d[lspec] += utils.radial_bin(np.real(act_lmaps[i][x] * act_lmaps[j][y].conj()), modlmap, bin_edges)

        # divide out the radial profile from the 2d spectra
        so_kspecs_2d[kspec] /= utils.interp1d_bins(_bin_edges, utils.radial_bin(so_kspecs_2d[kspec], modlmap, _bin_edges), bounds_error=False)(modlmap)
        act_kspecs_2d[kspec] /= utils.interp1d_bins(_bin_edges, utils.radial_bin(act_kspecs_2d[kspec], modlmap, _bin_edges), bounds_error=False)(modlmap)
        
        # normalize the 1d spectra by the mask and the analytical tf
        so_lspecs_1d[lspec] /= (so_w2 * 12)
        act_lspecs_1d[lspec] /= (act_w2 * 12)
        so_lspecs_1d[lspec] = np.divide(so_lspecs_1d[lspec], tf, out=np.zeros_like(so_lspecs_1d[lspec]), where=tf!=0)
        act_lspecs_1d[lspec] = np.divide(act_lspecs_1d[lspec], tf, out=np.zeros_like(act_lspecs_1d[lspec]), where=tf!=0)

In [ ]:
# i dont know why these spectra are off by a factor of 40, but clearly the 
# SO spectra are too noisy for any polarization
plt.plot(lb, so_lspecs_1d['EE'] * lb**2 / (2*np.pi), label='SO i1_f090 EE')
plt.plot(lb, act_lspecs_1d['EE'] * lb**2 / (2*np.pi), label='ACT pa5_f090 EE')
plt.legend()

In [ ]:
for i in range(3):
    enplot.pshow(so_wmask_nh * so_maps[0][i], ticks=5, downgrade=8, colorbar=True, range=[400, 200, 200][i])
    enplot.pshow(act_wmask_nh * act_maps[0][i], ticks=5, downgrade=8, colorbar=True, range=[400, 200, 200][i])

In [ ]:
enplot.pshow(enmap.ndmap(enmap.fftshift(act_kspecs_2d['UU']), lwcs)[(2880 - 72)//2:(2880 + 72)//2, (9600 - 240)//2:(9600 + 240)//2], ticks=[50, 90], upgrade=[6, 2], colorbar=True, min=-2, max=4)

In [ ]:
# since the 1d spectra aren't super useful, this does the same as above but
# just for SO 2d spectra. also we save compute by skipping pol crosses

so_map_dir = '/scratch/gpfs/SIMONSOBS/users/zatkins/projects/lat-iso/piso/tiger_dr6xbossn_20250804_split/maps/so/chervias'
mask_dir = '/scratch/gpfs/SIMONSOBS/users/zatkins/projects/lat-iso/piso/tiger_dr6xbossn_20250804_split/windows'

so_mapset = 'i6_f090'

so_maps = [enmap.read_map(f'{so_map_dir}/sky_map_{so_mapset}_bundle{i}_20250801.fits') * 1e6 for i in range(4)]
so_maps = enmap.samewcs(so_maps, so_maps[0])
so_wmask_nh = enmap.read_map(f'{mask_dir}/window_so_{so_mapset}_baseline_no_holes.fits')

_bin_edges = np.arange(0, 20000, 50) 
modlmap = so_maps.modlmap().astype(np.float32)
lwcs = enmap.lwcs(so_maps.shape, so_maps.wcs)

kpols = 'IQU'
so_kspecs_2d = {}

so_kmaps = enmap.fft(so_maps * so_wmask_nh)

for x in range(3):
    for y in range(x, 3):
        if x != y:
            continue
        
        kspec = kpols[x] + kpols[y]
        so_kspecs_2d[kspec] = 0

        for i in range(4):
            for j in range(i+1, 4):
                so_kspecs_2d[kspec] += np.real(so_kmaps[i][x] * so_kmaps[j][y].conj())
        
        so_kspecs_2d[kspec] /= utils.interp1d_bins(_bin_edges, utils.radial_bin(so_kspecs_2d[kspec], modlmap, _bin_edges), bounds_error=False)(modlmap)

enplot.pshow(enmap.ndmap(enmap.fftshift(so_kspecs_2d['II']), lwcs)[(2880 - 72)//2:(2880 + 72)//2, (9600 - 240)//2:(9600 + 240)//2], ticks=[50, 90], upgrade=[6, 2], colorbar=True, min=-2, max=4)
enplot.pshow(enmap.ndmap(enmap.fftshift(so_kspecs_2d['QQ']), lwcs)[(2880 - 72)//2:(2880 + 72)//2, (9600 - 240)//2:(9600 + 240)//2], ticks=[50, 90], upgrade=[6, 2], colorbar=True, min=-2, max=4)
enplot.pshow(enmap.ndmap(enmap.fftshift(so_kspecs_2d['UU']), lwcs)[(2880 - 72)//2:(2880 + 72)//2, (9600 - 240)//2:(9600 + 240)//2], ticks=[50, 90], upgrade=[6, 2], colorbar=True, min=-2, max=4)